# RAG (Retrieval-Augmented Generation) with Agno

RAG gives agents a **persistent knowledge base** they can search at runtime — enabling responses grounded in your own data rather than the model's training data alone.

## How it works
```
Your Documents
     │
     ▼ (embed)
Vector Database  ◄──── Agent searches at query time
     │                        │
     └── relevant chunks ─────┘
                              │
                    LLM generates grounded answer
```

## Agno Knowledge Base API (vs phidata)

| phidata | agno (current) |
|---------|------|
| `phi.agent.AgentKnowledge` | `agno.knowledge.knowledge.Knowledge` |
| `phi.vectordb.lancedb.LanceDb` | `agno.vectordb.lancedb.LanceDb` |
| `phi.knowledge.pdf.PDFUrlKnowledgeBase` | `Knowledge(...).insert(url=...)` |
| `phi.knowledge.csv.CSVKnowledgeBase` | `Knowledge(...).insert(path=...)` |
| `phi.embedder.sentence_transformer` | `agno.knowledge.embedder.sentence_transformer` |
| `phi.vectordb.chroma.ChromaDb` | `agno.vectordb.chroma.ChromaDb` |

Agno consolidated the format-specific knowledge base classes (`PDFUrlKnowledgeBase`, `CSVKnowledgeBase`, `AgentKnowledge`, ...) into a single `Knowledge` class with one `insert()` method that dispatches on `text_content=`, `url=`, or `path=` and auto-selects a reader from the file extension.

## We cover 4 RAG patterns:
1. **Text RAG** — load raw text into a vector store
2. **PDF RAG** — index a PDF document for Q&A
3. **CSV RAG** — query tabular data semantically
4. **Hybrid RAG** — combine RAG with web search for best-of-both

In [1]:
! pip install agno openai lancedb "sentence-transformers==3.0.1" "numpy<2" torch pypdf chromadb ddgs aiofiles python-dotenv


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
for var in ['OPENAI_API_KEY', 'OPENAI_BASE_URL', 'OPENAI_API_BASE']:
    if os.getenv(var):
        print(f"⚠️  Removing conflicting {var}")
        del os.environ[var]

os.environ["OPENAI_API_KEY"] = os.getenv("OPEN_ROUTER_KEY")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
print("✅ Environment ready")

✅ Environment ready


---
## Part 1 — Text RAG

The simplest form: load raw text strings into a vector database and query them.
The agent will search the knowledge base before answering.

In [3]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.knowledge.knowledge import Knowledge
from agno.vectordb.lancedb import LanceDb, SearchType
from agno.knowledge.embedder.sentence_transformer import SentenceTransformerEmbedder

# Create a vector database backed by LanceDB (local, no server needed)
vector_db = LanceDb(
    table_name="agentic_patterns_kb",
    uri="tmp/lancedb",
    search_type=SearchType.vector,
    embedder=SentenceTransformerEmbedder(id="sentence-transformers/all-MiniLM-L6-v2"),  # free, local embedder
)

# Build the knowledge base
knowledge_base = Knowledge(vector_db=vector_db)

# Load text facts into the knowledge base
texts = [
    "The Single-Agent pattern uses one AI model with tools and a system prompt. Best for simple multi-step tasks and prototypes.",
    "The Sequential pattern executes agents in a fixed linear order. No AI model is used for orchestration. Best for ETL pipelines.",
    "The Parallel pattern runs multiple agents concurrently and aggregates their outputs. Reduces latency for independent sub-tasks.",
    "The Coordinator pattern uses an AI model to dynamically route tasks to specialized subagents. More flexible than hardcoded patterns.",
    "The Hierarchical pattern decomposes tasks across multiple levels (root → manager → worker). Best for complex ambiguous problems.",
    "The Swarm pattern uses all-to-all communication between agents with no central coordinator. Best for creative debate and consensus building.",
    "The ReAct pattern loops through Thought → Action → Observation until an exit condition is met. Excellent for dynamic, open-ended research.",
    "The Review and Critique pattern uses a Generator and a Critic in a loop. The critic approves or sends back for revision.",
    "The Human-in-the-Loop pattern inserts human review checkpoints. Required for high-stakes, safety-critical, or compliance-sensitive workflows.",
    "The Custom Logic pattern uses Python code for orchestration, mixing multiple patterns with conditional branching.",
    "Agno is the evolution of phidata. Key additions: Team primitive for multi-agent, Agno OS playground, ~10,000 agents/sec throughput.",
    "In Agno, Team(mode='coordinate') implements the Coordinator pattern. Team(mode='parallel') implements the Parallel pattern. Team(mode='collaborate') implements the Swarm pattern.",
]

for text in texts:
    knowledge_base.insert(text_content=text)

print(f"✅ Loaded {len(texts)} text chunks into the knowledge base")

/Users/raamraam/Documents/GenAIEngineering-Cohort5/Week12/agno_env/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


INFO Creating table: agentic_patterns_kb

[2026-06-18T13:45:34Z WARN  lance::dataset::write::insert] No existing dataset at /Users/raamraam/Documents/GenAIEngineering-Cohort5/Week12/day_1/tmp/lancedb/agentic_patterns_kb.lance, it will be created


INFO Adding content from The Single

INFO Selecting reader for extension: Text

...

INFO Deleted 1 records with content_hash '71988c4d8e0803ba4519f0b2864c1331c14a1890bf8694e251379177bfedb5c3' from   
     table 'agentic_patterns_kb'.

INFO Adding content from In Agno, T

INFO Selecting reader for extension: Text

INFO Deleted 1 records with content_hash '71988c4d8e0803ba4519f0b2864c1331c14a1890bf8694e251379177bfedb5c3' from   
     table 'agentic_patterns_kb'.

✅ Loaded 12 text chunks into the knowledge base


In [4]:
# Create a RAG agent — it will search the knowledge base before answering
rag_agent = Agent(
    name="Agentic Patterns RAG Agent",
    model=OpenAIChat(id="google/gemini-2.5-flash"),
    knowledge=knowledge_base,
    search_knowledge=True,  # agent decides when to search
    instructions=[
        "Always search your knowledge base before answering questions about agentic AI patterns.",
        "If the knowledge base doesn't have the answer, say so — don't hallucinate.",
        "Quote from the knowledge base when relevant.",
    ],
    markdown=True,
)

# Test — agent should search the KB and answer from it
rag_agent.print_response(
    "What is the difference between the Coordinator pattern and the Parallel pattern?",
    stream=True,
)

Output()

INFO Found 1 documents

In [5]:
# This should be answered from the KB (Agno facts we loaded)
rag_agent.print_response(
    "How does Agno differ from phidata, and which Team modes are available?",
    stream=True,
)

Output()

INFO Found 1 documents

In [6]:
# This is NOT in the KB — agent should say so
rag_agent.print_response(
    "What is the current stock price of NVIDIA?",
    stream=True,
)

Output()

---
## Part 2 — PDF RAG

Index a PDF document and answer questions from it.
The agent automatically chunks, embeds, and stores the PDF content.

In [7]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.knowledge.knowledge import Knowledge
from agno.vectordb.lancedb import LanceDb, SearchType
from agno.knowledge.embedder.sentence_transformer import SentenceTransformerEmbedder
from IPython.display import Markdown, display

# Build a knowledge base from a PDF URL
pdf_knowledge_base = Knowledge(
    vector_db=LanceDb(
        table_name="thai_recipes",
        uri="tmp/lancedb",
        search_type=SearchType.vector,
        embedder=SentenceTransformerEmbedder(id="sentence-transformers/all-MiniLM-L6-v2"),
    ),
)

# Load the PDF — skip_if_exists avoids re-embedding on reruns (data is persisted in LanceDB)
print("📄 Loading PDF into vector database...")
pdf_knowledge_base.insert(
    url="https://phi-public.s3.amazonaws.com/recipes/ThaiRecipes.pdf",
    skip_if_exists=True,
)
print("✅ PDF indexed")

INFO Creating table: thai_recipes

📄 Loading PDF into vector database...


[2026-06-18T13:46:09Z WARN  lance::dataset::write::insert] No existing dataset at /Users/raamraam/Documents/GenAIEngineering-Cohort5/Week12/day_1/tmp/lancedb/thai_recipes.lance, it will be created


INFO Adding content from URL https://phi-public.s3.amazonaws.com/recipes/ThaiRecipes.pdf

✅ PDF indexed


In [8]:
# PDF RAG agent
recipe_agent = Agent(
    name="Thai Recipes Chef Agent",
    model=OpenAIChat(id="google/gemini-2.5-flash"),
    knowledge=pdf_knowledge_base,
    search_knowledge=True,
    instructions=[
        "You are a Thai cuisine expert. Only answer from the provided recipe book.",
        "If a recipe is not in the book, say so.",
        "Always include ingredient quantities and step-by-step instructions.",
    ],
    markdown=True,
)

response = recipe_agent.run(
    "Restrict your response to the provided knowledge base. "
    "How do I make chicken and galangal in coconut milk soup?"
)
display(Markdown(response.content))

INFO Found 10 documents

## Tom Kha Gai: Chicken and Galangal in Coconut Milk Soup

This recipe yields one serving and takes approximately 30 minutes to prepare and cook.

**Ingredients:**

*   150 grams chicken, cut into bite-size pieces
*   50 grams sliced young galangal
*   100 grams lightly crushed lemongrass, julienned
*   100 grams straw mushrooms
*   250 grams coconut milk
*   100 grams chicken stock
*   3 tbsp lime juice
*   3 tbsp fish sauce
*   2 leaves kaffir lime, shredded
*   1-2 bird’s eye chilies, pounded
*   3 leaves coriander

**Directions:**

1.  Bring the chicken stock and coconut milk to a slow boil. Add galangal, lemongrass, chicken, and mushrooms.
2.  When the soup returns to a boil, season it with fish sauce.
3.  Wait until the chicken is cooked, then add the kaffir lime leaves and bird's eye chilies.
4.  Remove the pot from heat and add lime juice.
5.  Garnish with coriander leaves.

**Tips:**

*   Keep the heat low throughout the cooking process. High heat will cause the oil in the coconut milk to separate and rise to the top.
*   If using mature galangal, reduce the amount.
*   Lime juice becomes more aromatic when added after the pot is removed from heat.
*   Reduce the amount of chilies for a milder taste.

This dish is rich in Vitamin A, B1, B2, B6, C, E, K, Folate, Copper, Iron, and Phosphorus.

In [9]:
# Follow-up question — should still search the KB
recipe_agent.print_response(
    "What can I substitute for galangal if I can't find it?",
    stream=True,
)

Output()

INFO Found 10 documents

---
## Part 3 — CSV RAG

Index a CSV file for semantic search — useful for natural language queries over tabular data.

> **Note:** For pure structured queries (SQL-style), `CsvTools` + `PythonTools` (used in the Knowledge Graph notebook) is better. CSV RAG shines for semantic, free-text search over rows.

In [10]:
import httpx
from pathlib import Path

# Download IMDB CSV (skip if already exists)
imdb_path = Path("imdb.csv")
if not imdb_path.exists():
    print("📥 Downloading IMDB dataset...")
    r = httpx.get("https://phidata-public.s3.amazonaws.com/demo_data/IMDB-Movie-Data.csv")
    imdb_path.write_bytes(r.content)
    print(f"✅ Saved to {imdb_path}")
else:
    print(f"✅ IMDB CSV already exists at {imdb_path}")

📥 Downloading IMDB dataset...
✅ Saved to imdb.csv


In [11]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.knowledge.knowledge import Knowledge
from agno.vectordb.lancedb import LanceDb, SearchType
from agno.knowledge.embedder.sentence_transformer import SentenceTransformerEmbedder
from IPython.display import Markdown, display

# Build knowledge base from the IMDB CSV
csv_knowledge_base = Knowledge(
    vector_db=LanceDb(
        table_name="imdb_movies",
        uri="tmp/lancedb",
        search_type=SearchType.vector,
        embedder=SentenceTransformerEmbedder(id="sentence-transformers/all-MiniLM-L6-v2"),
    ),
)

print("📊 Indexing IMDB CSV into vector database...")
csv_knowledge_base.insert(path="imdb.csv", skip_if_exists=True)
print("✅ IMDB dataset indexed")

INFO Creating table: imdb_movies

📊 Indexing IMDB CSV into vector database...


[2026-06-18T13:46:36Z WARN  lance::dataset::write::insert] No existing dataset at /Users/raamraam/Documents/GenAIEngineering-Cohort5/Week12/day_1/tmp/lancedb/imdb_movies.lance, it will be created


INFO Adding content from path, 4d72b1c4-9bbc-5fc8-8fbc-094a6020fe7f, None, imdb.csv, None

✅ IMDB dataset indexed


In [12]:
# CSV RAG agent — semantic search over movie rows
movie_rag_agent = Agent(
    name="IMDB Movie RAG Agent",
    model=OpenAIChat(id="google/gemini-2.5-flash"),
    knowledge=csv_knowledge_base,
    search_knowledge=True,
    instructions=[
        "Answer questions about movies from the IMDB dataset.",
        "Search your knowledge base for relevant movies before answering.",
        "Present results in a clean markdown table when listing movies.",
        "Always include rating, year, and director when available.",
    ],
    markdown=True,
)

# Semantic queries — these work better than SQL for vague descriptions
queries = [
    "List action movies with high ratings",
    "Find sci-fi movies that are also comedies",
    "Which Christopher Nolan movies are in the dataset?",
]

for query in queries:
    print(f"\n{'─'*50}")
    print(f"❓ {query}")
    print('─'*50)
    response = movie_rag_agent.run(query)
    display(Markdown(response.content))


──────────────────────────────────────────────────
❓ List action movies with high ratings
──────────────────────────────────────────────────


INFO Found 10 documents

It looks like you're looking for action movies with high ratings! Here's a list based on your criteria: 

| Title | Rating | Year | Director |
|---|---|---|---|
| Deadpool | 8 | 2016 | Tim Miller |
| Rogue One | 7.9 | 2016 | Gareth Edwards |
| Grindhouse | 7.6 | 2007 | Quentin Tarantino and Robert Rodriguez |


──────────────────────────────────────────────────
❓ Find sci-fi movies that are also comedies
──────────────────────────────────────────────────


INFO Found 10 documents

Here are some sci-fi comedies:

| Rating | Year | Director | Title |
|---|---|---|---|
| 7 | 2013 | Edgar Wright | The World's End |
| 7 | 2011 | Greg Mottola | Paul |


──────────────────────────────────────────────────
❓ Which Christopher Nolan movies are in the dataset?
──────────────────────────────────────────────────


INFO Found 10 documents

| Title | Rating | Year | Director |
|---|---|---|---|
| Interstellar | 8.6 | 2014 | Christopher Nolan |
| Inception | 8.8 | 2010 | Christopher Nolan |
| The Prestige | 8.5 | 2006 | Christopher Nolan |

---
## Part 4 — Hybrid RAG

Combine a **knowledge base** (your private data) with **web search** (current public data).
The agent decides which source to use based on the query.

In [13]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.knowledge.knowledge import Knowledge
from agno.vectordb.lancedb import LanceDb, SearchType
from agno.knowledge.embedder.sentence_transformer import SentenceTransformerEmbedder
from agno.tools.duckduckgo import DuckDuckGoTools

# Internal KB: company-specific facts not on the internet
internal_kb = Knowledge(
    vector_db=LanceDb(
        table_name="company_internal",
        uri="tmp/lancedb",
        search_type=SearchType.vector,
        embedder=SentenceTransformerEmbedder(id="sentence-transformers/all-MiniLM-L6-v2"),
    )
)

# Load internal facts (things web search can't find)
internal_facts = [
    "Our product 'AgnoAssist v2.0' was launched on March 1, 2025 with support for 12 GCP agentic patterns.",
    "Pricing: Starter plan $49/month (1M tokens), Pro plan $199/month (10M tokens), Enterprise custom pricing.",
    "SLA: 99.9% uptime guarantee. Support response time: 4 hours for Pro, 1 hour for Enterprise.",
    "Internal roadmap Q3 2025: Add swarm pattern templates, MCP server marketplace, and voice interface.",
    "Known issue: LanceDB table names with hyphens cause errors — use underscores instead.",
    "Our team is based in Bangalore, India. CEO: Mali R. Founded 2024.",
]

for fact in internal_facts:
    internal_kb.insert(text_content=fact)

# Hybrid agent: internal KB + web search
hybrid_agent = Agent(
    name="Hybrid RAG + Web Agent",
    model=OpenAIChat(id="google/gemini-2.5-flash"),
    knowledge=internal_kb,
    search_knowledge=True,  # search internal KB
    tools=[DuckDuckGoTools()],  # also search the web
    instructions=[
        "For questions about our product, pricing, or internal policies — search your knowledge base.",
        "For questions about general AI trends, competitors, or current events — search the web.",
        "Always tell the user which source you used: 'From internal KB:' or 'From web search:'.",
        "Never mix internal facts with web facts without clearly labeling them.",
    ],
    markdown=True,
)

# Internal question — should use KB
print("\n📁 Internal question (should use KB):")
hybrid_agent.print_response(
    "What are our product pricing plans and what's on the Q3 roadmap?",
    stream=True,
)

INFO Creating table: company_internal

[2026-06-18T13:47:46Z WARN  lance::dataset::write::insert] No existing dataset at /Users/raamraam/Documents/GenAIEngineering-Cohort5/Week12/day_1/tmp/lancedb/company_internal.lance, it will be created


INFO Adding content from Our produc

INFO Selecting reader for extension: Text

INFO Adding content from Pricing: S

...

INFO Selecting reader for extension: Text

INFO Deleted 1 records with content_hash '71988c4d8e0803ba4519f0b2864c1331c14a1890bf8694e251379177bfedb5c3' from   
     table 'company_internal'.


📁 Internal question (should use KB):


Output()

INFO Found 1 documents

In [14]:
# External question — should use web search
print("\n🌐 External question (should use web search):")
hybrid_agent.print_response(
    "What are the latest developments in agentic AI frameworks in 2025?",
    stream=True,
)


🌐 External question (should use web search):


Output()

In [15]:
# Mixed question — should use both
print("\n🔀 Mixed question (should use both):")
hybrid_agent.print_response(
    "How does our product compare to what competitors are launching in the agentic AI space?",
    stream=True,
)


🔀 Mixed question (should use both):


Output()

INFO Found 1 documents

WARNING  Could not run function web_search(query=...): No results found.

ERROR    No results found.                                                                                         
         Traceback (most recent call last):                                                                        
           File                                                                                                    
         "/Users/raamraam/Documents/GenAIEngineering-Cohort5/Week12/agno_env/lib/python3.11/site-packages/agno/tool
         s/function.py", line 1068, in execute                                                                     
             result = self.function.entrypoint(**entrypoint_args, **self.arguments)  # type: ignore                
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                
           File                                                                                                    
         "/Users/raamraam/Documents/GenAIEngineering-Cohort5/Week12/agno_env/lib/python3.11/site-packages/pydantic/
         _internal/_validate_call.py", line 40, in wrapper_function                                                
             return wrapper(*args, **kwargs)                                                                       
                    ^^^^^^^^^^^^^^^^^^^^^^^^                                                                       
           File                                                                                                    
         "/Users/raamraam/Documents/GenAIEngineering-Cohort5/Week12/agno_env/lib/python3.11/site-packages/pydantic/
         _internal/_validate_call.py", line 137, in __call__                                                       
             res = self.__pydantic_validator__.validate_python(pydantic_core.ArgsKwargs(args, kwargs))             
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^             
           File                                                                                                    
         "/Users/raamraam/Documents/GenAIEngineering-Cohort5/Week12/agno_env/lib/python3.11/site-packages/agno/tool
         s/websearch.py", line 98, in web_search                                                                   
             results = ddgs.text(**search_kwargs)                                                                  
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^                                                                  
           File                                                                                                    
         "/Users/raamraam/Documents/GenAIEngineering-Cohort5/Week12/agno_env/lib/python3.11/site-packages/ddgs/ddgs
         .py", line 458, in text                                                                                   
             return self._search_sync("text", query, **kwargs)                                                     
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                     
           File                                                                                                    
         "/Users/raamraam/Documents/GenAIEngineering-Cohort5/Week12/agno_env/lib/python3.11/site-packages/ddgs/ddgs
         .py", line 454, in _search_sync                                                                           
             raise DDGSException(err or "No results found.")                                                       
         ddgs.exceptions.DDGSException: No results found.

---
## RAG Summary

| RAG Type | Knowledge Source | Agno API | Best For |
|----------|-----------------|------------|----------|
| Text RAG | Raw strings | `Knowledge.insert(text_content=...)` | Facts, policies, FAQs |
| PDF RAG | PDF documents | `Knowledge.insert(url=...)` | Reports, manuals, books |
| CSV RAG | Tabular CSV | `Knowledge.insert(path=...)` | Semantic search over rows |
| Hybrid RAG | KB + web search | `Knowledge` + `DuckDuckGoTools` | Private + public data |

## Vector Database Options in Agno

| DB | Class | Best For |
|----|-------|----------|
| LanceDB | `LanceDb` | Local dev, serverless, fast |
| ChromaDB | `ChromaDb` | Local/remote, easy setup |
| PgVector | `PgVector` | Production PostgreSQL |
| Pinecone | `Pinecone` | Managed cloud production |

👉 **Next**: `8_knowledge_graph.ipynb` — build a knowledge graph from IMDB data and query it with second-order reasoning.